# Session Analysis
All trades, fills, markouts, and PnL from today's reward farming sessions.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timezone

DB_PATH = 'trades.db'
db = sqlite3.connect(DB_PATH)
df = pd.read_sql('SELECT * FROM trades ORDER BY ts', db)
db.close()

df['dt'] = pd.to_datetime(df['dt'])
df['cost'] = df['price'] * df['size']

fills = df[df['side'].str.contains('FILL')].copy()
tape = df[~df['side'].str.contains('FILL')].copy()

fills['is_buy'] = fills['side'] == 'FILL_BUY'
fills['is_tp'] = fills['side'] == 'FILL_SELL_TP'
fills['is_sl'] = fills['side'].str.contains('SL')
fills['token_type'] = np.where(fills['price'] < 0.5, 'underdog', 'favorite')

print(f'Total trades: {len(df)}')
print(f'Our fills: {len(fills)} ({fills["is_buy"].sum()} buys, {fills["is_tp"].sum()} TP sells, {fills["is_sl"].sum()} SL sells)')
print(f'Market tape: {len(tape)}')
print(f'Time range: {df.dt.min().strftime("%H:%M")} - {df.dt.max().strftime("%H:%M")} UTC')
print(f'Markets: {fills["market"].nunique()}')

In [ ]:
# ── All fills as a clean DataFrame ──
fill_summary = fills[['dt','market','side','price','size','cost','obi','mid','token_type']].copy()
fill_summary['dt_str'] = fill_summary['dt'].dt.strftime('%H:%M:%S')
fill_summary = fill_summary.sort_values('dt')
print(f'{len(fill_summary)} fills total')
fill_summary

In [ ]:
# ── PnL by market ──
pnl_data = []
for mkt, group in fills.groupby('market'):
    buys = group[group['is_buy']]
    sells = group[group['is_tp'] | group['is_sl']]
    buy_cost = buys['cost'].sum()
    buy_shares = buys['size'].sum()
    sell_rev = sells['cost'].sum()
    sell_shares = sells['size'].sum()
    avg_buy = buy_cost / buy_shares if buy_shares > 0 else 0
    avg_sell = sell_rev / sell_shares if sell_shares > 0 else 0
    realized = sell_rev - (avg_buy * sell_shares) if sell_shares > 0 else 0
    residual = buy_shares - sell_shares
    pnl_data.append({
        'market': mkt[:30],
        'buys': int(buy_shares),
        'avg_buy': round(avg_buy, 3),
        'buy_cost': round(buy_cost, 2),
        'sells': int(sell_shares),
        'avg_sell': round(avg_sell, 3),
        'sell_rev': round(sell_rev, 2),
        'realized_pnl': round(realized, 2),
        'residual': int(residual),
    })

pnl_df = pd.DataFrame(pnl_data)
print('Total realized PnL: $' + str(pnl_df['realized_pnl'].sum().round(2)))
print('Total residual shares:', pnl_df['residual'].sum())
print()
pnl_df

In [ ]:
# ── Markout computation ──
WINDOWS = [5, 10, 30, 60, 120, 300]

markout_rows = []
for _, fill in fills[fills['is_buy']].iterrows():
    if fill['price'] < 0.5:
        mt = tape[tape['price'] < 0.6]
    else:
        mt = tape[tape['price'] > 0.4]
    
    row = {
        'dt': fill['dt'],
        'market': fill['market'],
        'price': fill['price'],
        'size': fill['size'],
        'cost': fill['cost'],
        'obi': fill['obi'],
        'token_type': fill['token_type'],
    }
    for w in WINDOWS:
        future = mt[(mt['ts'] > fill['ts']) & (mt['ts'] <= fill['ts'] + w)]
        if len(future) > 0:
            fmid = np.average(future['price'], weights=future['size'])
            row[f'mo_{w}s'] = (fmid - fill['price']) * 100
            row[f'pnl_{w}s'] = (fmid - fill['price']) * fill['size']
        else:
            row[f'mo_{w}s'] = np.nan
            row[f'pnl_{w}s'] = np.nan
    markout_rows.append(row)

mo = pd.DataFrame(markout_rows)
print(f'Markouts computed for {len(mo)} BUY fills')
mo.head()

In [ ]:
# ── Chart 1: Price timeline + our fills (per market) ──
top_markets = fills[fills['is_buy']].groupby('market')['size'].sum().sort_values(ascending=False).head(4).index

fig, axes = plt.subplots(len(top_markets), 1, figsize=(18, 4*len(top_markets)))
if len(top_markets) == 1:
    axes = [axes]

for ax, mkt in zip(axes, top_markets):
    mkt_fills = fills[fills['market'] == mkt]
    avg_px = mkt_fills['price'].mean()
    mt = tape[tape['price'] < 0.6] if avg_px < 0.5 else tape[tape['price'] > 0.4]
    
    # Tape
    buys_t = mt[mt['side'] == 'BUY']
    sells_t = mt[mt['side'] == 'SELL']
    ax.scatter(buys_t['dt'], buys_t['price'], c='lightgreen', s=1, alpha=0.3, label='Tape BUY')
    ax.scatter(sells_t['dt'], sells_t['price'], c='lightcoral', s=1, alpha=0.3, label='Tape SELL')
    
    # Our buys
    our_buys = mkt_fills[mkt_fills['is_buy']]
    ax.scatter(our_buys['dt'], our_buys['price'], c='blue', s=our_buys['size'].clip(upper=500)/5+15,
              alpha=0.9, edgecolors='navy', zorder=5, label='Our BUY')
    
    # Our TP sells
    our_sells = mkt_fills[mkt_fills['is_tp']]
    if len(our_sells) > 0:
        ax.scatter(our_sells['dt'], our_sells['price'], c='lime', s=our_sells['size'].clip(upper=500)/5+15,
                  alpha=0.9, edgecolors='darkgreen', zorder=5, marker='^', label='Our TP SELL')
    
    # Running avg
    if len(our_buys) > 0:
        cum_cost = (our_buys['price'] * our_buys['size']).cumsum()
        cum_sz = our_buys['size'].cumsum()
        ax.plot(our_buys['dt'], cum_cost/cum_sz, 'b--', lw=1.5, label='Avg price')
    
    ax.set_title(f'{mkt[:40]} — {len(our_buys)} buys, {our_buys["size"].sum():.0f} shares', fontsize=11)
    ax.set_ylabel('Price')
    ax.legend(fontsize=7, loc='best')
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 2: Markout curves ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# By market
ax = axes[0]
for mkt, grp in mo.groupby('market'):
    avgs = [grp[f'mo_{w}s'].mean() for w in WINDOWS]
    ax.plot(WINDOWS, avgs, 'o-', label=mkt[:20], markersize=4)
ax.axhline(0, color='black', lw=0.5, ls='--')
ax.set_xlabel('Seconds after fill')
ax.set_ylabel('Markout (cents)')
ax.set_title('Markout by Market')
ax.legend(fontsize=6)
ax.grid(True, alpha=0.2)

# By token type
ax = axes[1]
for tt, grp in mo.groupby('token_type'):
    avgs = [grp[f'mo_{w}s'].mean() for w in WINDOWS]
    ax.plot(WINDOWS, avgs, 'o-', label=tt, markersize=6, lw=2)
ax.axhline(0, color='black', lw=0.5, ls='--')
ax.set_xlabel('Seconds after fill')
ax.set_ylabel('Markout (cents)')
ax.set_title('Underdog vs Favorite')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)

# Cumulative PnL
ax = axes[2]
for w in [10, 30, 60]:
    valid = mo.dropna(subset=[f'pnl_{w}s']).sort_values('dt')
    ax.plot(valid['dt'], valid[f'pnl_{w}s'].cumsum(), label=f'{w}s markout')
ax.axhline(0, color='black', lw=0.5, ls='--')
ax.set_xlabel('Time')
ax.set_ylabel('Cumulative PnL ($)')
ax.set_title('Cumulative Markout PnL')
ax.legend()
ax.grid(True, alpha=0.2)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 3: OBI at fill time vs markout ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

obi_fills = mo[mo['obi'] > 0].copy()

if len(obi_fills) > 3:
    ax = axes[0]
    ax.scatter(obi_fills['obi'], obi_fills['mo_10s'], c='steelblue', alpha=0.6, s=obi_fills['size']*2)
    ax.axhline(0, color='black', lw=0.5, ls='--')
    ax.set_xlabel('OBI at fill time')
    ax.set_ylabel('10s Markout (cents)')
    ax.set_title('OBI vs 10s Markout (size = bubble size)')
    ax.grid(True, alpha=0.2)
    
    ax = axes[1]
    ax.scatter(obi_fills['obi'], obi_fills['mo_60s'], c='coral', alpha=0.6, s=obi_fills['size']*2)
    ax.axhline(0, color='black', lw=0.5, ls='--')
    ax.set_xlabel('OBI at fill time')
    ax.set_ylabel('60s Markout (cents)')
    ax.set_title('OBI vs 60s Markout')
    ax.grid(True, alpha=0.2)
else:
    axes[0].text(0.5, 0.5, 'Not enough OBI data yet', ha='center', va='center', fontsize=14)
    axes[1].text(0.5, 0.5, 'Run more sessions with OBI logging', ha='center', va='center', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 4: Fill size + price distributions ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

buy_fills = fills[fills['is_buy']]

ax = axes[0][0]
ax.hist(buy_fills['size'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(buy_fills['size'].median(), color='red', ls='--', label=f'Median: {buy_fills["size"].median():.0f}')
ax.set_xlabel('Fill Size')
ax.set_title('Fill Size Distribution')
ax.legend()
ax.grid(True, alpha=0.2)

ax = axes[0][1]
ax.hist(buy_fills['price'], bins=20, color='coral', edgecolor='white', alpha=0.8)
ax.set_xlabel('Fill Price')
ax.set_title('Fill Price Distribution')
ax.grid(True, alpha=0.2)

ax = axes[1][0]
buy_fills.groupby('market')['size'].sum().sort_values().plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('Total Shares')
ax.set_title('Volume by Market')
ax.grid(True, alpha=0.2)

ax = axes[1][1]
buy_fills.groupby('market')['cost'].sum().sort_values().plot.barh(ax=ax, color='coral')
ax.set_xlabel('Total Cost ($)')
ax.set_title('Dollar Exposure by Market')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 5: Cumulative position over time per market ──
fig, ax = plt.subplots(figsize=(16, 6))

for mkt in fills['market'].unique():
    mkt_fills = fills[fills['market'] == mkt].sort_values('dt')
    pos = []
    running = 0
    for _, r in mkt_fills.iterrows():
        if r['is_buy']:
            running += r['size']
        else:
            running -= r['size']
        pos.append(running)
    ax.step(mkt_fills['dt'], pos, where='post', label=mkt[:20], lw=1.5)

ax.axhline(0, color='black', lw=0.5, ls='--')
ax.set_ylabel('Net Position (shares)')
ax.set_xlabel('Time')
ax.set_title('Position Over Time by Market')
ax.legend(fontsize=7, loc='best')
ax.grid(True, alpha=0.2)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary stats ──
print('='*60)
print('SESSION SUMMARY')
print('='*60)

total_bought = buy_fills['cost'].sum()
tp_fills = fills[fills['is_tp']]
total_sold = tp_fills['cost'].sum()

print(f'Total bought:    ${total_bought:,.2f}  ({buy_fills["size"].sum():.0f} shares across {len(buy_fills)} fills)')
print(f'Total TP sold:   ${total_sold:,.2f}  ({tp_fills["size"].sum():.0f} shares across {len(tp_fills)} fills)')
print(f'Turnover:        ${total_bought + total_sold:,.2f}')
print()

# Pair cost
yes_b = buy_fills[buy_fills['price'] >= 0.5]
no_b = buy_fills[buy_fills['price'] < 0.5]
if len(yes_b) > 0 and len(no_b) > 0:
    yes_wavg = np.average(yes_b['price'], weights=yes_b['size'])
    no_wavg = np.average(no_b['price'], weights=no_b['size'])
    print(f'Favorite avg buy: {yes_wavg:.4f}')
    print(f'Underdog avg buy: {no_wavg:.4f}')
    print(f'Pair cost:        {yes_wavg + no_wavg:.4f} (1.00 = breakeven)')
    print(f'Spread cost:      {(yes_wavg + no_wavg - 1.0)*100:+.2f}¢ per pair')
print()

# Markout summary
mo_cols = [f'mo_{w}s' for w in WINDOWS]
print('Avg markout (cents):')
print((mo[mo_cols].mean() * 1).round(2).to_string())
print()

print('By token type:')
for tt, grp in mo.groupby('token_type'):
    avgs = grp[mo_cols].mean().round(2)
    print(f'  {tt}: {avgs.to_dict()}')
print()

# TP efficiency
if len(tp_fills) > 0:
    print(f'TP fill rate: {len(tp_fills)}/{len(buy_fills)} = {len(tp_fills)/len(buy_fills)*100:.0f}%')
    print(f'Avg time to TP: need timestamp tracking (TODO)')
print()
print(f'Realized PnL (TP exits): ${pnl_df["realized_pnl"].sum():.2f}')
print(f'Residual position: {pnl_df["residual"].sum():.0f} shares')
print(f'\nNote: rewards not included — check polymarket.com/rewards tomorrow')